In [131]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from tqdm import tqdm
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [132]:
def set_seed(seed=None):
    """Set random seed for reproducibility"""  
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    return seed

In [133]:
seed = set_seed(100)

## 1. Conditional diffusion model

In [134]:
sde_Nt = 126
sde_dt = 1 / 252
sde_T = 1 / 2
n_steps = 100

In [135]:
def make_beta_schedule(schedule='linear', n_timesteps=1000, start=1e-5, end=1e-2):
    if schedule == 'linear':
        betas = torch.linspace(start, end, n_timesteps)
    elif schedule == "quad":
        betas = torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
    elif schedule == "sigmoid":
        betas = torch.linspace(-6, 6, n_timesteps)
        betas = torch.sigmoid(betas) * (end - start) + start
    return betas
def extract(input, t, x):
    shape = x.shape
    out = torch.gather(input, 0, t.to(input.device))
    reshape = [t.shape[0]] + [1] * (len(shape) - 1)
    return out.reshape(*reshape)
betas = make_beta_schedule(schedule='linear', n_timesteps=n_steps, 
                           start=1e-5, end=1e-2).to(device=device)
alphas = 1 - betas
alphas_prod = torch.cumprod(alphas, 0)
alphas_prod_p = torch.cat([torch.tensor([1]).float().to(device), alphas_prod[:-1]], 0)
alphas_bar_sqrt = torch.sqrt(alphas_prod)
one_minus_alphas_bar_log = torch.log(1 - alphas_prod)
one_minus_alphas_bar_sqrt = torch.sqrt(1 - alphas_prod)

In [136]:
class ConditionalLinear(nn.Module):
    def __init__(self, num_in, num_out, n_steps):
        super(ConditionalLinear, self).__init__()
        self.num_out = num_out
        self.lin = nn.Linear(num_in, num_out)
        self.embed = nn.Embedding(n_steps, num_out)
        self.embed.weight.data.uniform_()

    def forward(self, x, t):
        out = self.lin(x)
        gamma = self.embed(t)
        out = gamma.view(-1, self.num_out) * out
        return out
class StepConditionalModel(nn.Module):
    def __init__(self, n_steps, x_dim):
        super(StepConditionalModel, self).__init__()
        self.lin1 = ConditionalLinear(x_dim + x_dim, 32, n_steps)
        self.lin2 = ConditionalLinear(32, 32, n_steps)
        self.lin3 = ConditionalLinear(32, 16, n_steps)
        self.lin4 = ConditionalLinear(16, 16, n_steps)
        self.lin5 = nn.Linear(16, x_dim)
    
    def forward(self, state, time, condition):
        x = torch.cat([state, condition], dim=1)
        x = F.softplus(self.lin1(x, time))
        x = F.softplus(self.lin2(x, time))
        x = F.softplus(self.lin3(x, time))
        x = F.softplus(self.lin4(x, time))
        return self.lin5(x)


In [137]:
### DDIM sampler, which is DDPM sampler when eta=1
#### Copied from Equation (12) in https://arxiv.org/abs/2010.02502
@torch.no_grad()
def p_sample_ddim(model_cond, x, t, condition, delta_t = 1, eta=0.1):
    t = torch.tensor([t], device=device).long()
    tp1 = torch.clamp(t - delta_t, 0, n_steps - 1)
    alphas_prod_t = extract(alphas_prod, t, x)
    alphas_prod_tp1 = extract(alphas_prod, tp1, x)
    beta_prod_t = 1 - extract(alphas_prod, t, x)
    beta_prod_tp1 = 1 - extract(alphas_prod, tp1, x)
    sigma2_t = (beta_prod_tp1 / beta_prod_t) * (1 - alphas_prod_t / alphas_prod_tp1)
    std_t = sigma2_t.sqrt() * eta
    pred_eps = model_cond(x, t, condition)
    pred_x0 = (x - beta_prod_t.sqrt() * pred_eps) / alphas_prod_t.sqrt()
    point_dir = (1 - alphas_prod_tp1 - (std_t ** 2)).sqrt() * pred_eps
    sample = alphas_prod_tp1.sqrt() * pred_x0 + point_dir + std_t * torch.randn_like(x)
    return (sample)
@torch.no_grad()
def p_sample_loop_ddim(model_cond, condition, delta_t=1, eta=0.1):
    cur_x = torch.randn_like(condition, device=device)
    for i in reversed(range(0, n_steps, delta_t)):
        cur_x = p_sample_ddim(model_cond, cur_x, i, condition, delta_t, eta)
    return cur_x
@torch.no_grad()
def p_sample_sde_ddim(model_cond, init_condition, delta_t=1, eta=0.1):
    condition = init_condition.clone().view(-1, 1)
    sde_sample_path = torch.empty(init_condition.shape[0], sde_Nt + 1, device=device)
    sde_sample_path[:, 0] = condition.view(-1)
    for i in range(sde_Nt):
        condition += p_sample_loop_ddim(model_cond[i], condition, delta_t, eta)
        sde_sample_path[:, i + 1] = condition.view(-1)
    return sde_sample_path.clamp(1e-3, 10)

## 2. q-Learning for MV problem

- Copied from its implimentation shown on JMLR page:
 https://www.dropbox.com/scl/fo/8tq6n5byw09cx3l7tgf5c/APHg7dCig7GlQeqbGngQUBA?dl=0&e=1&preview=precommitted_mv_ts_batch.m&rlkey=dy41aoy4y2v8pfc7rvyv3wd95
- The form of q and v functions and their gradients are defined in Section 7.1 of the paper: q-Learning in Continuous Time


In [138]:
x0 = 1.
r = 0.02
disc_factor = (-r * torch.arange(0, sde_Nt + 1) * sde_dt).exp().to(device)

In [139]:
@torch.no_grad()
def qvlearn(theta_v, theta_q, lagrange_w, xt_all, at_all, tt_all, reward_all, 
                x0, r_true, explorelambda):
    # grad_theta = [theta_v; theta_q] - [theta_v; theta_q];

    # v = @(t,x) (x - lagrange_w).^2 .* exp(-theta_v(3)*(T-t)) + theta_v(2)*(t.^2 - T^2) + theta_v(1) * (t-T);
    v = lambda t, x: ((x - lagrange_w)**2 * (-theta_v[2] * (sde_T - t)).exp()
                      + theta_v[1] * (t**2 - sde_T**2)
                      + theta_v[0] * (t - sde_T))

    v_all = v(tt_all, xt_all)
    # q = @(t,x,a) -0.5*exp(-theta_q(1) - theta_q(3) * (T - t))  .* (a + theta_q(2) .* (x - lagrange_w)).^2 - 0.5*explorelambda * ( log(2*pi*explorelambda) + theta_q(1) + theta_q(3) * (T - t));
    q = lambda t, x, a: (-0.5 * (-theta_q[0] - theta_q[2] * (sde_T - t)).exp() * ((a + theta_q[1] * (x - lagrange_w)) ** 2) 
                            - 0.5 * explorelambda * (np.log(2 * np.pi * explorelambda) + theta_q[0] + theta_q[2] * (sde_T - t)))
    
    q_all = q(tt_all, xt_all, at_all)

    # vdot = diff(v_all, 1, 2);
    vdot = torch.diff(v_all, dim=1)

    # g_2 = @(t,x,a) t.^2 - T^2;
    g_2 = lambda t, x, a: t**2 - sde_T**2
    # g_1 = @(t,x,a) t - T;
    g_1 = lambda t, x, a: t - sde_T
    # g_3 = @(t,x,a) (x - lagrange_w).^2 .* exp(-theta_v(3)*(T-t)) .* (t-T);
    g_3 = lambda t, x, a: (x - lagrange_w)**2 * (-theta_v[2] * (sde_T - t)).exp() * (t - sde_T)

    # g_4 = @(t,x,a) 0.5*exp(-theta_q(1) - theta_q(3) * (T - t))  .* (a + theta_q(2) .* (x - lagrange_w)).^2 - 0.5*explorelambda;
    g_4 = lambda t, x, a: 0.5 * (a + theta_q[1] * (x - lagrange_w)) ** 2 * (-theta_q[0] - theta_q[2] * (sde_T - t)).exp() - 0.5 * explorelambda

    #g_5 = @(t,x,a) -exp(-theta_q(1) - theta_q(3) * (T - t))  .* (a + theta_q(2) .* (x - lagrange_w)).* (x - lagrange_w);
    g_5 = lambda t, x, a: -(-theta_q[0] - theta_q[2] * (sde_T - t)).exp() * (a + theta_q[1] * (x - lagrange_w)) * (x - lagrange_w)
    
    # g_6 = @(t,x,a) 0.5*(T-t).*exp(-theta_q(1) - theta_q(3) * (T - t))  .* (a + theta_q(2) .* (x - lagrange_w)).^2 - 0.5*explorelambda*(T-t);
    g_6 = lambda t, x, a: 0.5 * (sde_T - t) * (-theta_q[0] - theta_q[2] * (sde_T - t)).exp() * (a + theta_q[1] * (x - lagrange_w)) ** 2 - 0.5 * explorelambda * (sde_T - t)

    g_1_all = g_1(tt_all, xt_all, at_all)
    g_2_all = g_2(tt_all, xt_all, at_all)
    g_3_all = g_3(tt_all, xt_all, at_all)
    g_4_all = g_4(tt_all, xt_all, at_all)
    g_5_all = g_5(tt_all, xt_all, at_all)
    g_6_all = g_6(tt_all, xt_all, at_all)

    grad_theta = torch.zeros(6, device=device)
    value_diff = vdot - q_all[:, :-1] * sde_dt
    grad_theta[0] = -torch.mean(torch.sum(value_diff * g_1_all[:, :-1], dim=1))
    grad_theta[1] = -torch.mean(torch.sum(value_diff * g_2_all[:, :-1], dim=1))
    grad_theta[2] = -torch.mean(torch.sum(value_diff * g_3_all[:, :-1], dim=1))
    grad_theta[3] = -torch.mean(torch.sum(value_diff * g_4_all[:, :-1], dim=1))
    grad_theta[4] = -torch.mean(torch.sum(value_diff * g_5_all[:, :-1], dim=1))
    grad_theta[5] = -torch.mean(torch.sum(value_diff * g_6_all[:, :-1], dim=1))
    return grad_theta.clone().clamp(-1e-1, 1e-1)

In [140]:
@torch.no_grad()
def q_learning(stock_path, gen_path_func, x0, r, explorelambda, 
               batch_size, num_iter=1, report=False, theta_init=None, **kwargs):
    if theta_init is None:
        theta_init = torch.tensor([1, 1, 1, 0, -np.log(explorelambda), 0], device=device) 
    theta_v = theta_init[:3].clone()
    theta_q = theta_init[3:].clone()
    add_st = None
    temp_mean = 0
    lagrange_w = 0
    learned_strategy = torch.zeros((num_iter, 4), device=device)
    for k in tqdm(range(1, num_iter), disable=not report):
        st, add_st = gen_path_func(stock_path, batch_size, k, add_st, **kwargs)
        st *= disc_factor
        reward_path = torch.zeros(batch_size, sde_Nt, device=device) # zeros(m,nt);
        x_path = x0 * torch.ones(batch_size, sde_Nt + 1, device=device) # x0 * ones(m,nt+1);
        a_path = torch.zeros(batch_size, sde_Nt + 1, device=device) #zeros(m,nt+1);

        x_now = x0 * torch.ones(batch_size, device=device) # x0 * ones(m,1);
        policy_mean_now = -theta_q[1] * (x_now - lagrange_w) # -theta_q(2)*(x_now - lagrange_w);
        policy_sd_now = (explorelambda * (theta_q[0] + theta_q[2] * sde_T).exp()).sqrt()# sqrt(explorelambda * exp(theta_q(1) + theta_q(3) * T));
        action_now = policy_mean_now + policy_sd_now * torch.randn(batch_size, device=device)# policy_mean_now + policy_sd_now * randn(m,1);
        for t in range(sde_Nt - 1):
            a_path[:, t] = action_now # a_path(:,t) = action_now;
            # x_new = x_now + action_now * (np.exp(-r * (t + 1) * sde_dt) * st[:, t + 1] - np.exp(-r * t * sde_dt) * st[:, t]) / st[:, t] / np.exp(-r * t * sde_dt) # x_new = x_now + action_now .* (st(:,t+1)-st(:,t))./st(:,t);
            x_new = x_now + action_now * (st[:, t + 1] - st[:, t]) / st[:, t] # x_new = x_now + action_now .* (st(:,t+1)-st(:,t))./st(:,t);

            # policy_mean_new = -theta_q(2)*(x_new - lagrange_w);
            policy_mean_new = -theta_q[1] * (x_new - lagrange_w)
            # policy_sd_new = sqrt(explorelambda * exp(theta_q(1) + theta_q(3) * (T - t*dt)));
            policy_sd_new = (explorelambda * (theta_q[0] + theta_q[2] * (sde_T - t * sde_dt)).exp()).sqrt()
            # action_new = policy_mean_new + policy_sd_new * randn(m,1);
            action_new = policy_mean_new + policy_sd_new * torch.randn(batch_size, device=device)
            
            # x_path(:,t+1) = x_new;
            x_path[:, t + 1] = x_new

            # ent = explorelambda * 0.5 * ( log(2 * pi * policy_sd_new^2) + 1 );
            ent = explorelambda * 0.5 * (torch.log(2 * np.pi * policy_sd_new**2) + 1)

            # reward_path(:,t) = ent * ones(m,1);
            reward_path[:, t] = ent
            x_now = x_new
            action_now = action_new

        # a_path(:,nt+1) = action_now;
        a_path[:, -1] = action_now

        # temp_mean(k) = mean(x_now);
        temp_mean += x_now.mean()
        grad_theta = qvlearn(theta_v, theta_q, lagrange_w, x_path, a_path, 
                            (torch.ones(batch_size, 1)*torch.arange(0, sde_T + sde_dt, sde_dt)).to(device), #(0:dt:T),
                            reward_path, x0, r, explorelambda)

        # if grad_theta.abs().max() < 1e-1:
        # theta_v = theta_v + stepsize_td * grad_theta(1:3)/k^0.55;
        theta_v = theta_v + stepsize_td * grad_theta[:3].clone() / k**0.55
            # theta_q = theta_q + stepsize_td * grad_theta(4:6)/k^0.55;
        theta_q = theta_q + stepsize_td * grad_theta[3:].clone() / k**0.55
        if k % 10 == 0:
            lagrange_w = lagrange_w - stepsize_w * (temp_mean / 10 - target_z)
            temp_mean = 0
        # clean torch cache
        torch.cuda.empty_cache()
        learned_strategy[k, :3] = theta_q
        learned_strategy[k, 3] = lagrange_w
        
    return temp_mean, learned_strategy, theta_q, lagrange_w

In [141]:
@torch.no_grad()
def test_policy(policy_q, policy_w, stock_path):
    dis_stock_path = stock_path * disc_factor
    state = torch.ones(stock_path.shape[0], device=device) 
    action = -policy_q[1] * (state - policy_w) 
    for t in range(sde_Nt):
        state = state + action * (dis_stock_path[:, t + 1] - dis_stock_path[:, t]) / dis_stock_path[:, t]
        action = -policy_q[1] * (state - policy_w)
    print(f'mean return: {state.mean().item()}', f'variance: {state.var().item()}', 
          f'sharpe ratio: {((state.mean() - 1) / state.var().sqrt()).item()}')
    return state.mean(), state.var(), (state.mean() - 1) / state.var().sqrt()

## 3. Experiments

In [142]:
target_z = 1.2
data_dir = '/home/yinbinha/adapted_diffusion_model/data/sp500/'
model_cond = torch.load(f'{data_dir}ScoreDM_SDE_sp500.pt', map_location=device)
train_sde_path = torch.load(f'{data_dir}train_sde_path.pt', map_location=device)
train_price_path = torch.load(f'{data_dir}train_price_path.pt', map_location=device)
test_path = torch.load(f'{data_dir}test_path_2010_2019_seed{seed}.pt', map_location=device)

/tmp/ipykernel_3960911/614630321.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_cond = torch.load(f'{data_dir}ScoreDM_SDE_sp500.pt', map_location=device)
/tmp/ipy

### 3.1 Model-based Policy

#### 3.1.1 Estimation based on training price trajectory

In [143]:
path_np = train_price_path.cpu().detach().numpy()
log_ratio = np.diff(np.log(path_np / path_np[[0]]), axis=0)
est_sigma2 = log_ratio.var(ddof=1) / sde_dt
est_sigma = np.sqrt(est_sigma2)
est_mu = (log_ratio.mean() / sde_dt + est_sigma2 / 2)
est_mu = torch.tensor([est_mu]).to(device)
est_sigma = torch.tensor([est_sigma]).to(device)
est_rho = (est_mu - r) / est_sigma
est_w = (target_z * torch.exp((est_rho**2) * sde_T) - 1.) / (torch.exp((est_rho**2) * sde_T) - 1)
est_result = test_policy([0, est_rho / est_sigma, est_w], est_w, test_path)

mean return: 1.3529809869612932 variance: 0.42561130570534733 sharpe ratio: 0.5410591366482135


#### 3.1.2 Estimation based on the mixture of training and generated paths

In [144]:
mix_est_mu = []
mix_est_sigma = []
# gen_dir = '/home/yinbinha/adapted_diffusion_model/samples/dfm_train_sde_path_ts1767771452_seed42'
for _ in tqdm(range(10)):
    init_cond = torch.ones(360, 1).to(device)
    gen_sm_path = p_sample_sde_ddim(model_cond, init_cond, 1, 1)
    gen_sm_path = gen_sm_path.cpu().detach().numpy()
    # gen_sm_path = np.load(f'{gen_dir}/sample_batch1.npy')
    sm_log_ratio = np.diff(np.log(gen_sm_path / gen_sm_path[:, [0]]), axis=1).reshape(-1)
    mix_log_ratio = np.concatenate([log_ratio, sm_log_ratio], axis=0)
    est_sigma2_1 = log_ratio.var(ddof=1) / sde_dt
    est_sigma_1 = np.sqrt(est_sigma2_1)
    est_mu_1 = (mix_log_ratio.mean() / sde_dt + est_sigma2 / 2)
    mix_est_mu.append(est_mu_1)
    mix_est_sigma.append(est_sigma_1)
mix_est_mu = torch.tensor(mix_est_mu).mean().to(device)
mix_est_sigma = torch.tensor(mix_est_sigma).mean().to(device)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [01:31<00:00,  9.19s/it]


In [145]:
mix_est_rho = (mix_est_mu - r) / mix_est_sigma
mix_est_w = (target_z * torch.exp((mix_est_rho**2) * sde_T) - 1.) / (torch.exp((mix_est_rho**2) * sde_T) - 1)
mix_est_result = test_policy([0, mix_est_rho / mix_est_sigma, mix_est_w], 
                             mix_est_w, test_path)

mean return: 1.2304607596794401 variance: 0.16600953497401105 sharpe ratio: 0.565627443405707


#### 3.1.3 Estimation based on the mixture of training and generated paths (ours)

In [146]:
mix_est_mu = []
mix_est_sigma = []
gen_dir = '/home/yinbinha/adapted_diffusion_model/samples/dfm_train_sde_path_ts1768460485_seed42'
for _ in tqdm(range(1, 11)):
    gen_sm_path = np.load(f'{gen_dir}/sample_batch{_}.npy')
    gen_sm_path = gen_sm_path[~(gen_sm_path < 0).any(axis=1)]
    sm_log_ratio = np.diff(np.log(gen_sm_path / gen_sm_path[:, [0]]), axis=1).reshape(-1)
    mix_log_ratio = np.concatenate([log_ratio, sm_log_ratio], axis=0)
    est_sigma2_1 = log_ratio.var(ddof=1) / sde_dt
    est_sigma_1 = np.sqrt(est_sigma2_1)
    est_mu_1 = (mix_log_ratio.mean() / sde_dt + est_sigma2 / 2)
    mix_est_mu.append(est_mu_1)
    mix_est_sigma.append(est_sigma_1)
mix_est_mu = torch.tensor(mix_est_mu).mean().to(device)
mix_est_sigma = torch.tensor(mix_est_sigma).mean().to(device)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 1305.34it/s]


In [147]:
mix_est_rho = (mix_est_mu - r) / mix_est_sigma
mix_est_w = (target_z * torch.exp((mix_est_rho**2) * sde_T) - 1.) / (torch.exp((mix_est_rho**2) * sde_T) - 1)
mix_est_result = test_policy([0, mix_est_rho / mix_est_sigma, mix_est_w], 
                             mix_est_w, test_path)

mean return: 1.3381010820362043 variance: 0.38778466182317767 sharpe ratio: 0.5429392604210712


### 3.2 Model-free q-Learning method

#### 3.2.1 Purely split training path

In [148]:
def path_sample(stock_path, batch_size, iter_idx, add_st=None):
    index = torch.randint(0, stock_path.shape[0], (batch_size,)) 
    return stock_path[index].clone(), None

In [149]:
explorelambda = 0.00005
stepsize_td = 0.07# ; %0.0001;
stepsize_w = 0.2# ;
_, _, learned_q, learned_w = q_learning(
    train_sde_path, path_sample, x0, r, explorelambda, 40, 3000, True)
learn_result = test_policy(learned_q, learned_w, test_path)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2999/2999 [01:40<00:00, 29.98it/s]


mean return: 1.184210214024603 variance: 0.04426566740281597 sharpe ratio: 0.8755484809166749


#### 3.2.1 Bootstrap 

In [150]:
# Based on implentation of the q-learning paper
def start_point_sample(long_path, batch_size, iter_idx, add_st=None):
    sample = []
    start_idx = np.random.randint(0, long_path.shape[0] - sde_Nt - 1, batch_size)
    for idx in start_idx:
        sample.append(long_path[idx:idx + sde_Nt + 1] / long_path[idx])
    return torch.stack(sample, dim=0).to(device), None

In [151]:
explorelambda = 0.00003
stepsize_td = 0.1# ; %0.0001;
stepsize_w = 0.1# ;
_, _, start_point_bs_q, start_point_bs_w = q_learning(
    train_price_path, start_point_sample, 
    x0, r, explorelambda, 40, 3000, True)
start_point_bs_result = test_policy(start_point_bs_q, start_point_bs_w, test_path)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2999/2999 [01:40<00:00, 29.89it/s]

mean return: 1.1862984716574039 variance: 0.04145921948586421 sharpe ratio: 0.9149528856374415


#### 3.2.3 Generative model

In [152]:
def sdm_sample(stock_path, batch_size, iter_idx, gen_sm_path, gen_prop, gen_gap=5000):
    gen_size = int(max(batch_size * gen_prop + 1, batch_size - stock_path.shape[0])) 
    idx = iter_idx % gen_gap
    if idx == 1:
        init_cond = torch.ones(gen_gap * gen_size, 1).to(device)
        gen_sm_path = p_sample_sde_ddim(model_cond, init_cond, 1, 1)
     
    index = torch.randint(0, stock_path.shape[0], (batch_size - gen_size,)) # randsample(N,m,'true');
    gen_path = gen_sm_path[idx * gen_size:(idx + 1) * gen_size]
    return torch.cat([stock_path[index].clone()[:, :sde_Nt + 1], gen_path[:, :sde_Nt + 1]]), gen_sm_path

In [153]:
explorelambda = 0.00001
stepsize_td = 0.05
stepsize_w = 0.5
_, _, sdm_q, sdm_w = q_learning(train_sde_path, sdm_sample, x0, r, explorelambda, 
               80, 3000, True, gen_prop=0.5, gen_gap=1000)
sdm_result = test_policy(sdm_q, sdm_w, test_path)[-1]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2999/2999 [02:08<00:00, 23.30it/s]

mean return: 1.1748403958151472 variance: 0.030326568522743486 sharpe ratio: 1.0039917516527659


#### 3.2.5 Genetive model (ours)

In [154]:
def sdm_sample_static(stock_path, batch_size, iter_idx, add_st, gen_prop, full_gen_data):
    """
    Modified sampling function for pre-generated data.
    
    Args:
        stock_path: Real historical data.
        batch_size: Total batch size (e.g., 80).
        iter_idx: Current iteration (Ignored, essentially).
        add_st: Old buffer state (Ignored).
        gen_prop: Proportion of generated data (e.g., 0.5).
        full_gen_data: Your pre-loaded tensor of generated paths.
    """
    
    # 1. Calculate Split (e.g., 40 Real / 40 Gen)
    gen_size = int(batch_size * gen_prop)
    real_size = batch_size - gen_size

    # 2. Sample from REAL Data (Randomly)
    real_indices = torch.randint(0, stock_path.shape[0], (real_size,))
    real_batch = stock_path[real_indices]

    # 3. Sample from GENERATED Data (Randomly)
    # This replaces the complex 'gen_gap' and 'p_sample_sde_ddim' logic
    gen_indices = torch.randint(0, full_gen_data.shape[0], (gen_size,))
    gen_batch = full_gen_data[gen_indices].to(stock_path.device) 

    # 4. Mix them
    mixed_batch = torch.cat([
        real_batch[:, :sde_Nt + 1], 
        gen_batch[:, :sde_Nt + 1]
    ], dim=0)

    # Return mixed_batch and None (since we don't need to pass a buffer state anymore)
    return mixed_batch, None

In [155]:
import glob
import os
gen_dir = '/home/yinbinha/adapted_diffusion_model/samples/dfm_train_sde_path_ts1768460485_seed42'

# Folder path

# Get all batch files sorted by batch number
files = sorted(glob.glob(os.path.join(gen_dir, "sample_batch*.npy")))

# Load and combine
arrays = []
for f in files:
    arr = np.load(f)
    if arr.shape[1] != 127:
        raise ValueError(f"File {f} has unexpected shape {arr.shape}")
    arrays.append(arr)
    
my_generated_samples = np.concatenate(arrays, axis=0)

In [156]:
from functools import partial

# 1. Load Data
my_generated_samples = torch.from_numpy(my_generated_samples).to(device)

# 2. Create the Function
# We bind 'full_gen_data' to your loaded tensor
my_gen_func = partial(sdm_sample_static, full_gen_data=my_generated_samples)

# 3. Run Training
explorelambda = 0.00001
stepsize_td = 0.05
stepsize_w = 0.5

_, _, learned_q, learned_w = q_learning(
    stock_path=train_sde_path,
    gen_path_func=my_gen_func,  # Pass the modified function
    x0=x0, 
    r=r, 
    explorelambda=explorelambda, 
    batch_size=80,    # 40 Real + 40 Gen
    gen_prop=0.5,
    num_iter=3000, 
    report=True
)
sdm_result = test_policy(learned_q, learned_w, test_path)[-1]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2999/2999 [01:40<00:00, 29.88it/s]

mean return: 1.147432275975413 variance: 0.02108144187277874 sharpe ratio: 1.015412287128181
